# BERT Fine-tune Experiments

Тот же фича-инжиниринг, что и в ML_Experiments.ipynb, включая ALS.
**LaBSE fine-tuned on (vacancy, resume, target) pairs.
Сравнение: baseline LaBSE vs fine-tuned LaBSE + TF-IDF.**

In [1]:
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import mlflow
import optuna
import nltk
import pymorphy3

from tqdm.auto import tqdm
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix, vstack
from transformers import AutoTokenizer, AutoModel
from implicit.als import AlternatingLeastSquares
from optuna.integration.mlflow import MLflowCallback
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

import nltk
import pymorphy3
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess

warnings.simplefilter('ignore', FutureWarning)

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000
EXPERIMENT_NAME = "hr-ai-scout-bert-finetune"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")


In [3]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Device: {DEVICE}")

torch.manual_seed(RANDOM_STATE)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_STATE)

Device: mps


## 1. Load data

In [4]:
df = pd.read_csv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/total_df.csv')
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_salary_from,vacancy_salary_to,vacancy_salary_currency,vacancy_salary_gross,...,resume_education,resume_courses,resume_salary,resume_age,resume_total_experience,resume_experience_months,resume_location,resume_gender,resume_applicant_status,target
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Казанский Авиационный Институт'],NaN,NaN,65.0,19 лет,228.0,Москва,Мужчина,Рассматривает предложения,1
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,"['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...","['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...",NaN,43.0,17 лет 4 месяца,208.0,Москва,Мужчина,Рассматривает предложения,1
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Орский государственный педагогический инстит...,NaN,200 000 ₽ на руки,52.0,30 лет,360.0,Москва,Женщина,NaN,1
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Красноярский государственный университет'],NaN,500 000 ₽ на руки,56.0,29 лет 8 месяцев,356.0,Красноярск,Мужчина,Рассматривает предложения,1
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Белоруский Гос. Университет Информатики и Ра...,"['SAP CIS, SAP XI', 'Школа Логистики МАДИ', 'S...",NaN,48.0,25 лет 1 месяц,301.0,Moscow,Male,NaN,1


## 2. Preprocessing

*(скопировано из ML_Experiments.ipynb без изменений)*

In [5]:
t1 = df.shape[0]
df = df.dropna(subset=[
    "resume_education", "resume_last_experience_description",
    "resume_last_position", "resume_last_company_experience_period",
    "resume_total_experience", "resume_experience_months",
    "resume_location", "resume_specialization",
], how="all")
print('Удалено ', t1 - df.shape[0], ' строки')


Удалено  84  строки


In [6]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].notna()
              & df["resume_last_experience_description"].isna()
              & df["resume_last_position"].isna())]
df = df.loc[~(df["resume_total_experience"].isna()
              & df["resume_last_experience_description"].notna()
              & df["resume_last_position"].notna())]
print('Удалено ', t1 - df.shape[0], ' строк')


Удалено  1543  строк


In [7]:
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna('NDT')
df['resume_age'] = df['resume_age'].fillna(df['resume_age'].mean())
df['resume_experience_months'] = df['resume_experience_months'].fillna(0)
df = df.drop(['vacancy_salary_to', 'vacancy_salary_from',
              'vacancy_salary_currency', 'vacancy_salary_gross'], axis=1)


In [8]:
df['resume_salary_split'] = df['resume_salary'].apply(lambda x: x.split())
df['salary_int'] = df['resume_salary_split'].apply(
    lambda x: int(''.join(p for p in x if re.fullmatch(r'\d+', p)))
              if any(re.fullmatch(r'\d+', p) for p in x) else np.nan
)
currency_symbols = ['₽', '$', '€', '₴', '₸', '₼', '₾', 'Br', "so'm"]
rates_rub = {'₽':1.0,'$':80.85,'€':94.14,'₴':1.94,'₸':0.150,
             '₼':47.8,'₾':33.5,'Br':28.7,"so'm":0.0068}
df['currency_symbol'] = df['resume_salary_split'].apply(
    lambda x: next((s for s in x if s in currency_symbols), np.nan))
df['salary_converted'] = (df['salary_int'] * df['currency_symbol'].map(rates_rub)).fillna(0)
df['resume_salary'] = df['salary_converted']
df = df.drop(['resume_salary_split','salary_int','currency_symbol','salary_converted'], axis=1)


In [9]:
def experience_to_months(text):
    months = 0
    for pat in [r'(\d+)\s*год', r'(\d+)\s*лет']:
        m = re.search(pat, text)
        if m: months += int(m.group(1)) * 12
    m = re.search(r'(\d+)\s*месяц', text)
    if m: months += int(m.group(1))
    return months if months > 0 else np.nan

df['resume_last_company_experience_months'] = \
    df['resume_last_company_experience_period'].apply(experience_to_months)
df['resume_last_company_experience_months'] = \
    df['resume_last_company_experience_months'].fillna(0)


In [10]:
df = df[~(df.resume_salary > 1e7)]
df.loc[df['resume_experience_months'] > 720, 'resume_experience_months'] = 720
df.loc[df['resume_last_company_experience_months'] > 720, 'resume_last_company_experience_months'] = 720
df = df[~(df.resume_age > 90)]
df = df[~(df.resume_experience_months < df.resume_last_company_experience_months)]
df = df[~(df.resume_age < (df.resume_experience_months // 12) + 16)]

gender_map = {'Мужчина':'Мужчина','Male':'Мужчина','Женщина':'Женщина','Female':'Женщина'}
df['resume_gender'] = df['resume_gender'].apply(
    lambda x: gender_map.get(x, 'Неизвестно'))

print(f"Датасет после очистки: {df.shape}")


Датасет после очистки: (325543, 25)


## 3. Feature engineering (с TF-IDF)

In [11]:
# Признак совпадения локации
df['location_matching'] = (df['vacancy_area'] == df['resume_location']).astype(int)

# Количество навыков резюме в тексте вакансии
def resume_skill_count_in_vacancy(row):
    skills = row['resume_skills'].replace('[','').replace(']','').replace("'","").split(', ')
    return sum(1 for s in skills if s in row['vacancy_description'])

df['resume_skill_count_in_vacancy'] = df.apply(resume_skill_count_in_vacancy, axis=1)

# Доля слов последней должности, встречающихся в описании вакансии
def last_position_in_vacancy(row):
    bow = []
    for sep in [' ', '-', '_']:
        bow += row['resume_last_position'].split(sep=sep)
    bow = list(set(bow))
    return sum(1 for w in bow if w in row['vacancy_description']) / len(bow)

df['last_position_in_vacancy'] = df.apply(last_position_in_vacancy, axis=1)

print("Признаки добавлены: location_matching, resume_skill_count_in_vacancy, last_position_in_vacancy")


Признаки добавлены: location_matching, resume_skill_count_in_vacancy, last_position_in_vacancy


In [12]:
def preprocess_data(df):
    """Обработка пропущенных значений в текстовых полях"""
    print("Проверка пропущенных значений...")
    print(f"Пропуски в vacancy_description: {df['vacancy_description'].isna().sum()}")
    print(f"Пропуски в resume_last_experience_description: {df['resume_last_experience_description'].isna().sum()}")
    
    # Заполняем пропуски пустыми строками
    df['vacancy_description'] = df['vacancy_description'].fillna('')
    df['resume_last_experience_description'] = df['resume_last_experience_description'].fillna('')
    
    # Проверяем, что все значения теперь строковые
    df['vacancy_description'] = df['vacancy_description'].astype(str)
    df['resume_last_experience_description'] = df['resume_last_experience_description'].astype(str)
    
    return df

In [13]:
def save_results(df, output_file):
    """Сохранение результатов в CSV файл"""
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Результаты сохранены в файл: {output_file}")

In [14]:
def calculate_cosine_similarity(embeddings1, embeddings2):
    """Вычисление косинусного сходства между двумя наборами эмбеддингов"""
    similarities = []
    
    for i in tqdm(range(embeddings1.shape[0])):
        emb1_row = embeddings1[i]
        emb2_row = embeddings2[i]
        
        similarity = cosine_similarity(emb1_row, emb2_row)[0][0]
        similarities.append(similarity)
    
    return similarities

In [15]:
warnings.filterwarnings('ignore')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('taggers/averaged_perceptron_tagger_ru')
except LookupError:
    nltk.download('averaged_perceptron_tagger_ru')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

morph = pymorphy3.MorphAnalyzer()

[nltk_data] Error loading wordnet: <urlopen error [Errno 60] Operation
[nltk_data]     timed out>


In [16]:
def lemmatize_russian(tokens):
    """Лемматизация русских слов"""
    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0]  # Берем самый вероятный разбор
        lemmas.append(parsed.normal_form)
    return lemmas

In [17]:
def tokenize_and_lemmatize(text):
    """Токенизация текста с лемматизацией и удалением стоп-слов"""
    tokens = simple_preprocess(text, deacc=True, min_len=2)
    stop_words = set(stopwords.words('russian') + stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    lemmatized_tokens = lemmatize_russian(tokens)
    
    return lemmatized_tokens

In [18]:
def get_tfidf_embeddings(texts, vectorizer=None, fit=True):
    """Создание TF-IDF эмбеддингов для списка текстов с лемматизацией"""
    if fit:
        vectorizer = TfidfVectorizer(
            max_features=5000,
            min_df=2,
            max_df=0.8,
            ngram_range=(1, 2),
            tokenizer=tokenize_and_lemmatize,
            token_pattern=None,
            lowercase=False  # Уже сделано в токенизации
        )
        embeddings = vectorizer.fit_transform(texts)
    else:
        embeddings = vectorizer.transform(texts)
    
    return embeddings, vectorizer

In [19]:
def get_tfidf_vacancy_embeddings(df, vectorizer=None):
    """Создание эмбеддингов для уникальных вакансий с лемматизацией"""
    unique_vacancies = df[['vacancy_id', 'vacancy_description']].drop_duplicates()
    
    unique_embeddings, vectorizer = get_tfidf_embeddings(
        unique_vacancies['vacancy_description'].tolist(), 
        vectorizer=vectorizer, 
        fit=(vectorizer is None)
    )
    
    vacancy_embedding_dict = dict(zip(unique_vacancies['vacancy_id'], unique_embeddings))
    
    rows = []
    for vid in df['vacancy_id']:
        rows.append(vacancy_embedding_dict[vid])
    
    all_vacancy_embeddings = vstack(rows)
    
    return all_vacancy_embeddings, vectorizer

In [20]:
def process_similarity_scores_tfidf(df, vectorizer=None, fit=True):
    """Функция для вычисления схожести с использованием TF-IDF и лемматизации"""
    df = preprocess_data(df)
    
    print("Создание TF-IDF эмбеддингов для описаний опыта в резюме...")
    experience_embeddings, tfidf_vectorizer = get_tfidf_embeddings(df['resume_last_experience_description'].tolist(), vectorizer=vectorizer, fit=fit)
    
    print("Создание TF-IDF эмбеддингов для описаний вакансий...")
    vacancy_embeddings, _ = get_tfidf_vacancy_embeddings(df, vectorizer=tfidf_vectorizer)
    
    print("Вычисление косинусного сходства...")
    similarity_scores = calculate_cosine_similarity(vacancy_embeddings, experience_embeddings)
    
    df['similarity_score_tfidf'] = similarity_scores
    
    return df, tfidf_vectorizer

In [21]:
try:
    df_tfidf = pd.read_csv('description_df_with_scores_tfidf.csv')
except:
    df_tfidf = process_similarity_scores_tfidf(df.copy())
    save_results(df_tfidf, 'description_df_with_scores_tfidf.csv')

In [22]:
df = df.merge(df_tfidf)

In [23]:
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_description,resume_id,resume_title,resume_specialization,...,resume_location,resume_gender,resume_applicant_status,target,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,resume_skill_count,similarity_score_tfidf
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",6969174,ABAP-разработчик,"['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,76.0,1,3,0.666667,3,0.284047
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",9100077,"ABAP разработчик - SAP HCM, CRM, S/4HANA ERP(F...","['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,8.0,1,2,0.500000,2,0.308726
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",32644957,Разработчик ABAP,"['Программист, разработчик']",...,Москва,Женщина,NDT,1,136.0,1,1,0.000000,1,0.510093
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",27220466,ABAP-разработчик,"['Программист, разработчик']",...,Красноярск,Мужчина,Рассматривает предложения,1,135.0,0,2,0.333333,2,0.301062
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",7532708,ABAP разработчик. Senior ABAP Developer. SAP T...,"['Programmer, developer']",...,Moscow,Мужчина,NDT,1,0.0,0,2,0.600000,2,0.075429


## 4. BERT Similarity

Ключ оптимизации — кодируем только **уникальные** вакансии и резюме, затем маппим на все строки df.

In [24]:
def encode_texts(texts, tokenizer, model, batch_size=64, prefix=''):
    """
    Батчевое кодирование текстов в L2-нормированные эмбеддинги.
    Mean pooling по токенам, взвешенный attention mask.
    """
    if prefix:
        texts = [prefix + t for t in texts]

    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="    encoding"):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            out = model(**encoded)

        # Mean pooling
        token_emb = out.last_hidden_state                              # (B, T, H)
        mask = encoded['attention_mask'].unsqueeze(-1).float()         # (B, T, 1)
        pooled = (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        pooled = F.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)


In [25]:
def compute_bert_similarity(df, tokenizer, model, batch_size=64,
                             vacancy_prefix='', resume_prefix=''):
    """
    Cosine similarity между vacancy_description и resume_last_experience_description.
    Эмбеддинги вычисляются только для уникальных текстов.
    """
    df = df.copy()
    df['vacancy_description'] = df['vacancy_description'].fillna('').astype(str)
    df['resume_last_experience_description'] = \
        df['resume_last_experience_description'].fillna('').astype(str)

    # ── Уникальные вакансии ──────────────────────────────────────────
    unique_vac = df[['vacancy_id','vacancy_description']].drop_duplicates('vacancy_id')
    print(f"  Уникальных вакансий: {len(unique_vac)} / всего строк: {len(df)}")
    print("  Эмбеддинги вакансий...")
    vac_emb = encode_texts(unique_vac['vacancy_description'].tolist(),
                           tokenizer, model, batch_size, prefix=vacancy_prefix)
    vac_map = dict(zip(unique_vac['vacancy_id'], vac_emb))

    # ── Уникальные резюме ────────────────────────────────────────────
    unique_res = df[['resume_id','resume_last_experience_description']].drop_duplicates('resume_id')
    print(f"  Уникальных резюме: {len(unique_res)}")
    print("  Эмбеддинги резюме...")
    res_emb = encode_texts(unique_res['resume_last_experience_description'].tolist(),
                           tokenizer, model, batch_size, prefix=resume_prefix)
    res_map = dict(zip(unique_res['resume_id'], res_emb))

    # ── Попарное сходство (L2-норм → cosine = dot) ───────────────────
    sims = []
    for _, row in df.iterrows():
        v = vac_map.get(row['vacancy_id'])
        r = res_map.get(row['resume_id'])
        sims.append(float(np.dot(v, r)) if v is not None and r is not None else 0.0)

    return sims


In [26]:
# (model_name, vacancy_prefix, resume_prefix, batch_size)
BERT_MODELS = [
    ('cointegrated/LaBSE-en-ru',                                    '', '',           64),
    ('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', '', '',           64),
    ('ai-forever/sbert_large_nlu_ru',                               '', '',           32),
    ('intfloat/multilingual-e5-base',                   'query: ', 'passage: ',      32),
]


## 4.1 Кеш эмбеддингов в ClickHouse

Сохраняем вычисленные эмбеддинги в ClickHouse один раз.  
При следующем запуске — загружаем из базы, **не пересчитываем**.

In [27]:
from clickhouse_driver import Client
import os
from dotenv import load_dotenv

load_dotenv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/.env')

ch = Client(
    host=os.getenv('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.getenv('CLICKHOUSE_PORT', 9000)),
    user=os.getenv('CLICKHOUSE_USER', 'default'),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
    database=os.getenv('CLICKHOUSE_DATABASE', 'default'),
)
print("ClickHouse подключён")


ClickHouse подключён


In [28]:
def get_missing_ids(ids_needed: list, table: str, id_col: str,
                    model_name: str, clickhouse) -> list:
    """
    Возвращает те id из ids_needed, для которых в ClickHouse
    ещё нет эмбеддингов (по model_name).
    """
    if not ids_needed:
        return []
    str_ids = [str(i) for i in ids_needed]
    rows = clickhouse.execute(
        f"SELECT {id_col} FROM {table} "
        f"WHERE model_name = %(m)s AND {id_col} IN %(ids)s",
        {'m': model_name, 'ids': str_ids}
    )
    existing = {row[0] for row in rows}
    missing = [i for i in str_ids if i not in existing]
    print(f"  {table}: {len(existing)} в кеше, {len(missing)} новых из {len(str_ids)}")
    return missing


def save_embeddings_to_ch(emb_map: dict, id_col: str, table: str,
                           model_name: str, clickhouse):
    """Дописывает только новые эмбеддинги — существующие не удаляет."""
    rows = [
        (str(k), model_name, v.tolist())
        for k, v in emb_map.items()
    ]
    clickhouse.execute(
        f"INSERT INTO {table} ({id_col}, model_name, embedding) VALUES",
        rows,
    )
    print(f"  Сохранено {len(rows)} эмбеддингов → {table}")


def load_embeddings_from_ch(table: str, id_col: str, model_name: str,
                              clickhouse, ids: list = None) -> dict:
    """
    Загружает эмбеддинги из ClickHouse.
    ids — если передан, загружает только указанные id (экономит память).
    """
    params = {'m': model_name}
    query = f"SELECT {id_col}, embedding FROM {table} WHERE model_name = %(m)s"
    if ids:
        params['ids'] = [str(i) for i in ids]
        query += f" AND {id_col} IN %(ids)s"
    rows = clickhouse.execute(query, params)
    return {row[0]: np.array(row[1], dtype=np.float32) for row in rows}


In [29]:
bert_sim_cols = {}

for model_name, vac_prefix, res_prefix, bs in BERT_MODELS:
    short = model_name.split('/')[-1].replace('-', '_').lower()
    col   = f'sim_{short}'
    print(f"\n{'='*60}\n{model_name}\n{'='*60}")

    # Уникальные тексты текущего датасета
    unique_vac = (df[['vacancy_id', 'vacancy_description']]
                  .drop_duplicates('vacancy_id'))
    unique_res = (df[['resume_id', 'resume_last_experience_description']]
                  .drop_duplicates('resume_id'))

    all_vac_ids = unique_vac['vacancy_id'].tolist()
    all_res_ids = unique_res['resume_id'].tolist()

    # Проверяем, каких id нет в кеше
    missing_vac = get_missing_ids(all_vac_ids, 'vacancy_embeddings',
                                   'vacancy_id', model_name, ch)
    missing_res = get_missing_ids(all_res_ids, 'resume_embeddings',
                                   'resume_id', model_name, ch)

    # Загружаем BERT только если есть пропуски
    if missing_vac or missing_res:
        tokenizer  = AutoTokenizer.from_pretrained(model_name)
        bert_model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

        if missing_vac:
            texts = (unique_vac[unique_vac['vacancy_id'].astype(str).isin(missing_vac)]
                     ['vacancy_description'].fillna('').tolist())
            emb = encode_texts(texts, tokenizer, bert_model, bs, prefix=vac_prefix)
            save_embeddings_to_ch(dict(zip(missing_vac, emb)),
                                  'vacancy_id', 'vacancy_embeddings', model_name, ch)

        if missing_res:
            texts = (unique_res[unique_res['resume_id'].astype(str).isin(missing_res)]
                     ['resume_last_experience_description'].fillna('').tolist())
            emb = encode_texts(texts, tokenizer, bert_model, bs, prefix=res_prefix)
            save_embeddings_to_ch(dict(zip(missing_res, emb)),
                                  'resume_id', 'resume_embeddings', model_name, ch)

        del bert_model, tokenizer
        if DEVICE.type == 'mps':    torch.mps.empty_cache()
        elif DEVICE.type == 'cuda': torch.cuda.empty_cache()
    else:
        print("  Все эмбеддинги уже в кеше ClickHouse — загружаем")

    # Загружаем только нужные id текущего датасета
    vac_map = load_embeddings_from_ch('vacancy_embeddings', 'vacancy_id',
                                       model_name, ch, ids=all_vac_ids)
    res_map = load_embeddings_from_ch('resume_embeddings', 'resume_id',
                                       model_name, ch, ids=all_res_ids)

    # Косинусное сходство для каждой строки df
    df['vacancy_description'] = df['vacancy_description'].fillna('').astype(str)
    df['resume_last_experience_description'] = \
        df['resume_last_experience_description'].fillna('').astype(str)

    sims = [
        float(np.dot(vac_map[str(row.vacancy_id)], res_map[str(row.resume_id)]))
        if str(row.vacancy_id) in vac_map and str(row.resume_id) in res_map
        else 0.0
        for row in df.itertuples()
    ]
    df[col] = sims
    bert_sim_cols[model_name] = col
    print(f"  Среднее сходство: {df[col].mean():.4f}")

print("\nГотово:", list(bert_sim_cols.values()))



cointegrated/LaBSE-en-ru
  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20130 в кеше, 0 новых из 20130
  Все эмбеддинги уже в кеше ClickHouse — загружаем
  Среднее сходство: 0.6486

sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20130 в кеше, 0 новых из 20130
  Все эмбеддинги уже в кеше ClickHouse — загружаем
  Среднее сходство: 0.3834

ai-forever/sbert_large_nlu_ru
  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20130 в кеше, 0 новых из 20130
  Все эмбеддинги уже в кеше ClickHouse — загружаем
  Среднее сходство: 0.6443

intfloat/multilingual-e5-base
  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20130 в кеше, 0 новых из 20130
  Все эмбеддинги уже в кеше ClickHouse — загружаем
  Среднее сходство: 0.8039

Готово: ['sim_labse_en_ru', 'sim_paraphrase_multilingual_minilm_l12_v2', 'sim_sbert_large_nlu_ru', 'sim_multilingual_e5_base']

## 5. ALS

*(скопировано из ML_Experiments.ipynb без изменений)*

In [30]:
def create_interaction_matrix(df):
    unique_vacancies = df['vacancy_id'].unique().tolist()
    unique_resumes   = df['resume_id'].unique().tolist()
    id2vacancy = dict(enumerate(unique_vacancies))
    id2resume  = dict(enumerate(unique_resumes))
    vacancy2id = {v: k for k, v in id2vacancy.items()}
    resume2id  = {v: k for k, v in id2resume.items()}
    rows = [vacancy2id[v] for v in df['vacancy_id']]
    cols = [resume2id[r]  for r in df['resume_id']]
    matrix = csr_matrix((df['target'], (rows, cols)),
                        shape=(len(unique_vacancies), len(unique_resumes)),
                        dtype=np.float32)
    return matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes

def get_factors(interactions_matrix):
    als = AlternatingLeastSquares(
        factors=64, regularization=0.1, iterations=30,
        random_state=RANDOM_STATE, num_threads=0)
    als.fit(interactions_matrix.T)
    return als.item_factors, als.user_factors

def get_als_score(vacancy_id, resume_id):
    if vacancy_id not in vacancy2id or resume_id not in resume2id:
        return 0
    return float(np.dot(vacancy_factors[vacancy2id[vacancy_id]],
                         resume_factors[resume2id[resume_id]]))


## 6. Train / Test split + ALS score

In [31]:
# Базовые признаки (без similarity — добавим bert-вариант в цикле)
BASE_FEATURES = [
    'vacancy_area', 
    'vacancy_experience', 
    'vacancy_employment', 
    'vacancy_schedule',
    'resume_salary', 
    'resume_age', 
    'resume_experience_months', 
    'resume_location',
    'resume_gender', 
    'resume_applicant_status', 
    'resume_last_company_experience_months',
    'location_matching', 
    'resume_skill_count_in_vacancy', 
    'last_position_in_vacancy',
    'similarity_score_tfidf'
]

categorical_features = df[BASE_FEATURES].select_dtypes(exclude=np.number).columns.tolist()
numeric_features     = df[BASE_FEATURES].select_dtypes(include=np.number).columns.tolist()

X_base = df[BASE_FEATURES].copy()
y      = df['target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (257313, 15), Test: (64329, 15)


In [32]:
# ALS: обучаем на части train, чтобы избежать data leakage
X_train_als, _, y_train_als, _ = train_test_split(
    X_train, y_train, test_size=0.3, random_state=RANDOM_STATE, stratify=y_train)

train_als_interactions = df.loc[X_train_als.index, ['vacancy_id','resume_id','target']]
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, \
    unique_vacancies, unique_resumes = create_interaction_matrix(train_als_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

X_train['als_score'] = df.loc[X_train.index].apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), axis=1)

# Для test — ALS на полном train
train_interactions = df.loc[X_train.index, ['vacancy_id','resume_id','target']]
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, \
    unique_vacancies, unique_resumes = create_interaction_matrix(train_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

X_test['als_score'] = df.loc[X_test.index].apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), axis=1)

print(f"als_score в train (нули): {(X_train['als_score']==0).sum()}")
print(f"als_score в test  (нули): {(X_test['als_score']==0).sum()}")


  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

als_score в train (нули): 14
als_score в test  (нули): 0


## 7. Fine-tuning LaBSE-en-ru

Fine-tuning `cointegrated/LaBSE-en-ru` на обучающих парах `(vacancy_description, resume_last_experience_description, target)`.

**Важно:** используем только индексы `X_train`, чтобы не допустить утечки данных из тест-сета.

In [33]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
import os

FINETUNE_BASE_MODEL = 'cointegrated/LaBSE-en-ru'
FINETUNE_SAVE_PATH  = './models/LaBSE-hr-finetuned'
FINETUNE_MODEL_KEY  = 'LaBSE-hr-finetuned'
FT_SIM_COL          = 'sim_labse_hr_finetuned'

os.makedirs('./models', exist_ok=True)
print(f"Fine-tune: {FINETUNE_BASE_MODEL}  →  {FINETUNE_SAVE_PATH}")

Fine-tune: cointegrated/LaBSE-en-ru  →  ./models/LaBSE-hr-finetuned


In [34]:
# Пары ТОЛЬКО из тренировочного сета (leak-free)
ft_df = df.loc[X_train.index,
               ['vacancy_description', 'resume_last_experience_description', 'target']].copy()
ft_df['vacancy_description'] = ft_df['vacancy_description'].fillna('').astype(str)
ft_df['resume_last_experience_description'] = (
    ft_df['resume_last_experience_description'].fillna('').astype(str))

# Балансировка: не более N_MAX пар каждого класса
N_MAX = 15_000
pos = ft_df[ft_df['target'] == 1].sample(
    min(N_MAX, int(ft_df['target'].sum())), random_state=RANDOM_STATE)
neg = ft_df[ft_df['target'] == 0].sample(
    min(N_MAX, int((ft_df['target'] == 0).sum())), random_state=RANDOM_STATE)
ft_pairs = pd.concat([pos, neg]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Пар для fine-tuning: {len(ft_pairs)}  "
      f"({len(pos)} позитивных / {len(neg)} негативных)")

Пар для fine-tuning: 30000  (15000 позитивных / 15000 негативных)


In [47]:
import json

# ── Гиперпараметры ─────────────────────────────────────────────────────────
N_EPOCHS   = 3    # ← сколько эпох запустить В ЭТОМ СЕАНСЕ (не суммарно)
BATCH_SIZE = 16
LR         = 2e-5
MAX_LEN    = 256
STATE_FILE  = os.path.join(FINETUNE_SAVE_PATH, 'training_state.json')

def mean_pool(token_emb, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

# ── Загружаем чекпойнт или базовую модель ────────────────────────────────────
checkpoint_exists = os.path.isdir(FINETUNE_SAVE_PATH) and \
                    os.path.isfile(os.path.join(FINETUNE_SAVE_PATH, 'config.json'))

if checkpoint_exists:
    print(f"Чекпойнт найден: {FINETUNE_SAVE_PATH}")
    ft_tokenizer    = AutoTokenizer.from_pretrained(FINETUNE_SAVE_PATH)
    ft_model_train  = AutoModel.from_pretrained(FINETUNE_SAVE_PATH).to(DEVICE)
    state           = json.loads(open(STATE_FILE).read()) if os.path.isfile(STATE_FILE) else {}
    epochs_done     = state.get('epochs_completed', 0)
    print(f"Продолжаем с эпохи {epochs_done + 1}  (уже завершено: {epochs_done})")
else:
    print(f"Чекпойнт не найден — загружаем базовую модель: {FINETUNE_BASE_MODEL}")
    ft_tokenizer    = AutoTokenizer.from_pretrained(FINETUNE_BASE_MODEL)
    ft_model_train  = AutoModel.from_pretrained(FINETUNE_BASE_MODEL).to(DEVICE)
    epochs_done     = 0

ft_model_train.train()

# ── Оптимизатор и шедулер на N_EPOCHS шагов ──────────────────────────────────
optimizer = AdamW(ft_model_train.parameters(), lr=LR)
n_steps   = (len(ft_pairs) // BATCH_SIZE) * N_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.05 * n_steps)),
    num_training_steps=n_steps,
)

print(f"\nЗапускаем {N_EPOCHS} эпох(и)  |  Steps/epoch: {len(ft_pairs)//BATCH_SIZE}"
      f"  |  Device: {DEVICE}\n")

for epoch in range(N_EPOCHS):
    global_epoch = epochs_done + epoch + 1
    shuffled = ft_pairs.sample(frac=1, random_state=global_epoch).reset_index(drop=True)
    total_loss, n_batches = 0.0, 0

    for i in tqdm(range(0, len(shuffled), BATCH_SIZE),
                  desc=f"Epoch {global_epoch} (сеанс {epoch+1}/{N_EPOCHS})"):
        batch  = shuffled.iloc[i : i + BATCH_SIZE]
        texts1 = batch['vacancy_description'].tolist()
        texts2 = batch['resume_last_experience_description'].tolist()
        labels = torch.tensor(
            batch['target'].values * 2 - 1, dtype=torch.float).to(DEVICE)

        enc1 = ft_tokenizer(texts1, padding=True, truncation=True,
                            max_length=MAX_LEN, return_tensors='pt').to(DEVICE)
        enc2 = ft_tokenizer(texts2, padding=True, truncation=True,
                            max_length=MAX_LEN, return_tensors='pt').to(DEVICE)

        optimizer.zero_grad()
        emb1 = F.normalize(mean_pool(ft_model_train(**enc1).last_hidden_state,
                                     enc1['attention_mask']), p=2, dim=1)
        emb2 = F.normalize(mean_pool(ft_model_train(**enc2).last_hidden_state,
                                     enc2['attention_mask']), p=2, dim=1)

        loss = F.cosine_embedding_loss(emb1, emb2, labels, margin=0.5)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model_train.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item(); n_batches += 1

    avg_loss = total_loss / n_batches
    print(f"  Epoch {global_epoch}: avg_loss = {avg_loss:.4f}")

    # Сохраняем после каждой эпохи
    ft_model_train.eval()
    ft_model_train.save_pretrained(FINETUNE_SAVE_PATH)
    ft_tokenizer.save_pretrained(FINETUNE_SAVE_PATH)
    with open(STATE_FILE, 'w') as f:
        json.dump({'epochs_completed': global_epoch, 'last_avg_loss': avg_loss}, f, indent=2)
    ft_model_train.train()
    print(f"    ✓ чекпойнт сохранён (всего эпох: {global_epoch})")

ft_model_train.eval()
print(f"\nГотово. Модель сохранена: {FINETUNE_SAVE_PATH}")
print(f"Суммарно завершено эпох: {epochs_done + N_EPOCHS}")

del optimizer, scheduler, ft_model_train
import gc; gc.collect()
if DEVICE.type == 'mps':    torch.mps.empty_cache()
elif DEVICE.type == 'cuda': torch.cuda.empty_cache()

Чекпойнт найден: ./models/LaBSE-hr-finetuned


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Продолжаем с эпохи 4  (уже завершено: 3)

Запускаем 3 эпох(и)  |  Steps/epoch: 1875  |  Device: mps



Epoch 4 (сеанс 1/3):   0%|          | 0/1875 [00:00<?, ?it/s]

  Epoch 4: avg_loss = 0.0476


...

    ✓ чекпойнт сохранён (всего эпох: 5)


Epoch 6 (сеанс 3/3):   0%|          | 0/1875 [00:00<?, ?it/s]

  Epoch 6: avg_loss = 0.0360


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ чекпойнт сохранён (всего эпох: 6)

Готово. Модель сохранена: ./models/LaBSE-hr-finetuned
Суммарно завершено эпох: 6


In [48]:
# Вычисляем embeddings fine-tuned моделью (AutoModel, тот же encode_texts())
ft_tokenizer_infer = AutoTokenizer.from_pretrained(FINETUNE_SAVE_PATH)
ft_model_infer     = AutoModel.from_pretrained(FINETUNE_SAVE_PATH).to(DEVICE).eval()

unique_vac = df[['vacancy_id', 'vacancy_description']].drop_duplicates('vacancy_id')
unique_res = df[['resume_id', 'resume_last_experience_description']].drop_duplicates('resume_id')
all_vac_ids = unique_vac['vacancy_id'].tolist()
all_res_ids = unique_res['resume_id'].tolist()

missing_vac = get_missing_ids(all_vac_ids, 'vacancy_embeddings',
                               'vacancy_id', FINETUNE_MODEL_KEY, ch)
missing_res = get_missing_ids(all_res_ids, 'resume_embeddings',
                               'resume_id', FINETUNE_MODEL_KEY, ch)

if missing_vac:
    texts = (unique_vac[unique_vac['vacancy_id'].astype(str).isin(missing_vac)]
             ['vacancy_description'].fillna('').tolist())
    emb = encode_texts(texts, ft_tokenizer_infer, ft_model_infer, batch_size=64)
    save_embeddings_to_ch(dict(zip(missing_vac, emb)),
                          'vacancy_id', 'vacancy_embeddings', FINETUNE_MODEL_KEY, ch)

if missing_res:
    texts = (unique_res[unique_res['resume_id'].astype(str).isin(missing_res)]
             ['resume_last_experience_description'].fillna('').tolist())
    emb = encode_texts(texts, ft_tokenizer_infer, ft_model_infer, batch_size=64)
    save_embeddings_to_ch(dict(zip(missing_res, emb)),
                          'resume_id', 'resume_embeddings', FINETUNE_MODEL_KEY, ch)

if not missing_vac and not missing_res:
    print("Все эмбеддинги уже в кеше ClickHouse")

vac_map_ft = load_embeddings_from_ch('vacancy_embeddings', 'vacancy_id',
                                      FINETUNE_MODEL_KEY, ch, ids=all_vac_ids)
res_map_ft = load_embeddings_from_ch('resume_embeddings', 'resume_id',
                                      FINETUNE_MODEL_KEY, ch, ids=all_res_ids)

df['vacancy_description'] = df['vacancy_description'].fillna('').astype(str)
df['resume_last_experience_description'] = (
    df['resume_last_experience_description'].fillna('').astype(str))

sims_ft = [
    float(np.dot(vac_map_ft[str(row.vacancy_id)], res_map_ft[str(row.resume_id)]))
    if str(row.vacancy_id) in vac_map_ft and str(row.resume_id) in res_map_ft else 0.0
    for row in df.itertuples()
]
df[FT_SIM_COL] = sims_ft
bert_sim_cols[FINETUNE_MODEL_KEY] = FT_SIM_COL

print(f"\nСравнение similarity (mean / std):")
print(f"  LaBSE baseline:    {df['sim_labse_en_ru'].mean():.4f} / {df['sim_labse_en_ru'].std():.4f}")
print(f"  LaBSE fine-tuned:  {df[FT_SIM_COL].mean():.4f} / {df[FT_SIM_COL].std():.4f}")

del ft_tokenizer_infer, ft_model_infer, vac_map_ft, res_map_ft
import gc; gc.collect()
if DEVICE.type == 'mps':    torch.mps.empty_cache()
elif DEVICE.type == 'cuda': torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Error on localhost:9000 ping: Unexpected EOF while reading bytes
Connection was closed, reconnecting.
Error on socket shutdown: [Errno 57] Socket is not connected


  vacancy_embeddings: 3409 в кеше, 0 новых из 3409
  resume_embeddings: 20130 в кеше, 0 новых из 20130
Все эмбеддинги уже в кеше ClickHouse

Сравнение similarity (mean / std):
  LaBSE baseline:    0.6486 / 0.1118
  LaBSE fine-tuned:  0.5404 / 0.2033


## 8. Metrics

In [49]:
def calculate_metrics(df_test: pd.DataFrame) -> pd.DataFrame:
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    vacancy_ids = df_test['vacancy_id'].unique()
    
    for vacancy_id in vacancy_ids:
        mask = df_test['vacancy_id'] == vacancy_id
        y_true = df_test.loc[mask, 'target'].values
        y_score = df_test.loc[mask, 'y_pred_proba'].values
        
        if len(y_true) <= 1:
            continue
        
        y_true_2d = y_true.reshape(1, -1)
        y_score_2d = y_score.reshape(1, -1)
        
        ndcg = ndcg_score(y_true_2d, y_score_2d)
        ndcg_scores.append(ndcg)
        
        y_pred_binary = (y_score >= 0.5).astype(int)
        
        precision = precision_score(y_true, y_pred_binary, zero_division=0)
        recall = recall_score(y_true, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
    
    if ndcg_scores:
        print(f"Средний NDCG: {np.mean(ndcg_scores):.4f}")
        print(f"Средний Precision: {np.mean(precision_scores):.4f}")
        print(f"Средний Recall: {np.mean(recall_scores):.4f}")
        print(f"Средний F1-Score: {np.mean(f1_scores):.4f}")

        return np.mean(ndcg_scores), np.mean(precision_scores), np.mean(recall_scores), np.mean(f1_scores)
    else:
        print("Недостаточно данных для расчета метрик")

        return None, None, None, None

## 9. CatBoost + ALS + TF-IDF + BERT Fine-tuned

Запускаем CatBoost (Optuna, 10 trials) для каждого BERT-варианта:
4 базовых модели + fine-tuned LaBSE. Логируем в MLflow.

In [50]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

STUDY_DB_NAME = "sqlite:///local.study.finetune.db"


In [51]:
def run_cat_bert(model_name, sim_col):
    short = model_name.split('/')[-1].replace('-', '_').lower()
    cat_bert = categorical_features

    X_tr = X_train[BASE_FEATURES + ['als_score']].copy()
    X_tr[sim_col] = df.loc[X_train.index, sim_col].values

    X_te = X_test[BASE_FEATURES + ['als_score']].copy()
    X_te[sim_col] = df.loc[X_test.index, sim_col].values

    def objective_catboost(trial: optuna.Trial) -> float:
        params = {
            'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
            'model__depth': trial.suggest_int('depth', 3, 10),
            'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
        }
        
        pipeline_catboost = Pipeline([
            ('preprocessing', ColumnTransformer([
                ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
            ], remainder='passthrough')),
            ('model', CatBoostClassifier(
                random_state=RANDOM_STATE, 
                verbose=0, 
                auto_class_weights='Balanced'
            ))
        ])
        
        pipeline_catboost.set_params(**params)
        
        kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        
        for train_idx, val_idx in kfold.split(X_tr, y_train):
            X_fold_train, X_fold_val = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            pipeline_catboost.fit(X_fold_train, y_fold_train)
            y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
            
            df_val = df.loc[X_fold_val.index].copy()
            df_val['y_pred_proba'] = y_pred_proba[:, 1]
            
            ndcg, _, _, _ = calculate_metrics(df_val)
            
            trial.set_user_attr('pipeline_params', params)
        
        return ndcg

    RUN_NAME_OPTUNE_CATBOOST = f'CatBoostClassifier_optuna_als_tfidf_{short}'

    with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
        run_id_catboost = run.info.run_id

    STUDY_NAME_CATBOOST = f'CatBoostClassifier_optuna_als_tfidf_{short}'

    mlflc_catboost = MLflowCallback(
        tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
        metric_name="NDCG",
        create_experiment=False,
        mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
    )

    study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=False)

    study_catboost.optimize(objective_catboost, 
                            n_trials=10, 
                            callbacks=[mlflc_catboost]
                           )
    
    best_params_catboost = study_catboost.best_params
    print(f"Number of finished trials: {len(study_catboost.trials)}")
    print(f"Best params CatBoost: {best_params_catboost}")

    pipeline_catboost_best = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
    pipeline_catboost_best.fit(X_tr, y_train)
    
    y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_te)
    
    df_test_catboost = df.loc[X_te.index].copy()
    df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]
    
    ndcg_catboost_als, precision_catboost_als, recall_catboost_als, f1_catboost_als = calculate_metrics(df_test_catboost)
    metrics_catboost_als = {
        'NDCG': ndcg_catboost_als,
        'Precision': precision_catboost_als,
        'Recall': recall_catboost_als,
        'F1': f1_catboost_als
    }

    RUN_NAME_CATBOOST = f"best_optuna_catboost_als_tfidf_{short}"
    REGISTRY_MODEL_NAME_CATBOOST = f"best_optuna_catboost_als_tfidf_{short}"
    
    signature = mlflow.models.infer_signature(X_te, y_test)
    input_example = X_te[:10]
    code_paths = ["BERT_Experiments.ipynb"]
    
    with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
        run_id = run.info.run_id
        
        catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                                 artifact_path=f'best_optuna_catboost_als_tfidf_{short}',
                                                 registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                                 input_example=input_example,
                                                 code_paths=code_paths,
                                                 await_registration_for=60
                                                )
        mlflow.log_metrics(metrics_catboost_als)
        mlflow.log_params(best_params_catboost)

    return {'Model': model_name, 'sim_col': sim_col, 'Pipeline': pipeline_catboost_best, **metrics_catboost_als}


In [52]:
all_results = []

for model_name, _, _, _ in BERT_MODELS:
    sim_col = bert_sim_cols[model_name]
    print(f"\n{'='*60}\nЭксперимент: {model_name}\n{'='*60}")
    try:
        result = run_cat_bert(model_name, sim_col)
        all_results.append(result)
    except Exception as e:
        print(f"[ОШИБКА] {e}")
        all_results.append({'Model': model_name, 'sim_col': sim_col,
                             'NDCG': None, 'Pipeline': None})
        break


[I 2026-05-07 18:27:38,822] Using an existing study with name 'CatBoostClassifier_optuna_als_tfidf_labse_en_ru' instead of creating a new one.



Эксперимент: cointegrated/LaBSE-en-ru
🏃 View run CatBoostClassifier_optuna_als_tfidf_labse_en_ru at: http://127.0.0.1:5000/#/experiments/3/runs/eecd333ebbe84e2a9a8f2a8b1a4e8c14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Средний NDCG: 0.8678
Средний Precision: 0.7453
Средний Recall: 0.8544
Средний F1-Score: 0.7813
Средний NDCG: 0.8581
Средний Precision: 0.7364
Средний Recall: 0.8432
Средний F1-Score: 0.7719


[I 2026-05-07 18:28:01,522] Trial 20 finished with value: 0.8638827749253956 and parameters: {'iterations': 700, 'depth': 7, 'learning_rate': 0.017674158674824185, 'l2_leaf_reg': 1.711229251678547}. Best is trial 10 with value: 0.8642627323681974.


Средний NDCG: 0.8639
Средний Precision: 0.7459
Средний Recall: 0.8476
Средний F1-Score: 0.7796
🏃 View run 20 at: http://127.0.0.1:5000/#/experiments/3/runs/1b9c3d169c824324b3ab03bfece24399
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Средний NDCG: 0.8678
Средний Precision: 0.7546
Средний Recall: 0.8549
Средний F1-Score: 0.7877
Средний NDCG: 0.8576
Средний Precision: 0.7431
Средний Recall: 0.8432
Средний F1-Score: 0.7762


[I 2026-05-07 18:28:20,056] Trial 21 finished with value: 0.8642724199307286 and parameters: {'iterations': 400, 'depth': 7, 'learning_rate': 0.04354885641073019, 'l2_leaf_reg': 2.115241692081686}. Best is trial 21 with value: 0.8642724199307286.


...

[I 2026-05-07 19:30:10,969] Trial 29 finished with value: 0.8644217431693935 and parameters: {'iterations': 1000, 'depth': 10, 'learning_rate': 0.022295026826100826, 'l2_leaf_reg': 8.320385146180074}. Best is trial 29 with value: 0.8644217431693935.


Средний NDCG: 0.8644
Средний Precision: 0.7770
Средний Recall: 0.8426
Средний F1-Score: 0.7969
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/3/runs/f7c12236861749f9844997566625a2d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Number of finished trials: 30
Best params CatBoost: {'iterations': 1000, 'depth': 10, 'learning_rate': 0.022295026826100826, 'l2_leaf_reg': 8.320385146180074}
Средний NDCG: 0.7817
Средний Precision: 0.6935
Средний Recall: 0.7562
Средний F1-Score: 0.7105


Registered model 'best_optuna_catboost_als_tfidf_multilingual_e5_base' already exists. Creating a new version of this model...
2026/05/07 19:48:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost_als_tfidf_multilingual_e5_base, version 4


🏃 View run best_optuna_catboost_als_tfidf_multilingual_e5_base at: http://127.0.0.1:5000/#/experiments/3/runs/755665c1174941f18166484ce109f9ad
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Created version '4' of model 'best_optuna_catboost_als_tfidf_multilingual_e5_base'.


In [53]:
# ── Fine-tuned LaBSE через тот же run_cat_bert ──────────────────────────────
print(f"\n{'='*60}\nЭксперимент: {FINETUNE_MODEL_KEY}\n{'='*60}")
try:
    result_ft = run_cat_bert(FINETUNE_MODEL_KEY, FT_SIM_COL)
    all_results.append(result_ft)
except Exception as e:
    import traceback; traceback.print_exc()
    all_results.append({'Model': FINETUNE_MODEL_KEY, 'sim_col': FT_SIM_COL,
                         'NDCG': None, 'Pipeline': None})

[I 2026-05-08 10:27:29,998] Using an existing study with name 'CatBoostClassifier_optuna_als_tfidf_labse_hr_finetuned' instead of creating a new one.



Эксперимент: LaBSE-hr-finetuned
🏃 View run CatBoostClassifier_optuna_als_tfidf_labse_hr_finetuned at: http://127.0.0.1:5000/#/experiments/3/runs/442df1c213a647b1bc4db281e811cc5e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Средний NDCG: 0.8679
Средний Precision: 0.7750
Средний Recall: 0.8505
Средний F1-Score: 0.7993
Средний NDCG: 0.8570
Средний Precision: 0.7630
Средний Recall: 0.8398
Средний F1-Score: 0.7875


[I 2026-05-08 10:27:49,750] Trial 20 finished with value: 0.8630234152401935 and parameters: {'iterations': 700, 'depth': 5, 'learning_rate': 0.27047297227177763, 'l2_leaf_reg': 7.421533288534982}. Best is trial 19 with value: 0.8638628691145794.


Средний NDCG: 0.8630
Средний Precision: 0.7712
Средний Recall: 0.8467
Средний F1-Score: 0.7955
🏃 View run 20 at: http://127.0.0.1:5000/#/experiments/3/runs/d2ac3e92344b4adbbae8c53e6ff02b99
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Средний NDCG: 0.8683
Средний Precision: 0.7851
Средний Recall: 0.8496
Средний F1-Score: 0.8048
Средний NDCG: 0.8577
Средний Precision: 0.7711
Средний Recall: 0.8388
Средний F1-Score: 0.7921


[I 2026-05-08 10:28:14,188] Trial 21 finished with value: 0.8635593538357554 and parameters: {'iterations': 750, 'depth': 8, 'learning_rate': 0.062291392814073504, 'l2_leaf_reg': 7.545259731002635}. Best is trial 19 with value: 0.8638628691145794.


...

[I 2026-05-08 10:31:52,434] Trial 29 finished with value: 0.8635431163404715 and parameters: {'iterations': 600, 'depth': 10, 'learning_rate': 0.11233442131435559, 'l2_leaf_reg': 1.0894822629120313}. Best is trial 23 with value: 0.863911806268177.


Средний NDCG: 0.8635
Средний Precision: 0.7980
Средний Recall: 0.8322
Средний F1-Score: 0.8045
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/3/runs/e420ecb01af849c9899d39d68d15608e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Number of finished trials: 30
Best params CatBoost: {'iterations': 900, 'depth': 9, 'learning_rate': 0.07798542753997621, 'l2_leaf_reg': 3.1716799500493154}
Средний NDCG: 0.7806
Средний Precision: 0.7051
Средний Recall: 0.7516
Средний F1-Score: 0.7158


Registered model 'best_optuna_catboost_als_tfidf_labse_hr_finetuned' already exists. Creating a new version of this model...
2026/05/08 10:32:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost_als_tfidf_labse_hr_finetuned, version 3


🏃 View run best_optuna_catboost_als_tfidf_labse_hr_finetuned at: http://127.0.0.1:5000/#/experiments/3/runs/b91d3b6821a24e58bc93e2c0ac4a9d29
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Created version '3' of model 'best_optuna_catboost_als_tfidf_labse_hr_finetuned'.


## 10. Результаты

In [54]:
NDCG_TFIDF_BASELINE = 0.7817  # LightGBM + ALS + TF-IDF из ML_Experiments.ipynb

results_df = pd.DataFrame([
    {'Model': r['Model'], 'NDCG': r['NDCG'],
     'Precision': r.get('Precision'), 'Recall': r.get('Recall'), 'F1': r.get('F1')}
    for r in all_results
])
results_df['Delta vs TF-IDF'] = results_df['NDCG'] - NDCG_TFIDF_BASELINE
results_df = results_df.sort_values('NDCG', ascending=False).reset_index(drop=True)

print(f"Базовая линия TF-IDF: NDCG = {NDCG_TFIDF_BASELINE}")
print()
print(results_df[['Model','NDCG','Delta vs TF-IDF']].to_string(index=False))


Базовая линия TF-IDF: NDCG = 0.7817

                                                      Model     NDCG  Delta vs TF-IDF
                                   cointegrated/LaBSE-en-ru 0.782071         0.000371
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 0.781989         0.000289
                              intfloat/multilingual-e5-base 0.781686        -0.000014
                              ai-forever/sbert_large_nlu_ru 0.781220        -0.000480
                                         LaBSE-hr-finetuned 0.780581        -0.001119


In [55]:
valid = [r for r in all_results if r.get('NDCG') is not None]
if valid:
    best = max(valid, key=lambda r: r['NDCG'])
    if best['NDCG'] > NDCG_TFIDF_BASELINE:
        fname = f"pipeline_catboost_finetune_{best['sim_col']}.pkl"
        with open(fname, 'wb') as f:
            pickle.dump(best['Pipeline'], f)
        print(f"Лучший пайплайн сохранён: {fname}")
        print(f"NDCG: {best['NDCG']:.4f}  (TF-IDF: {NDCG_TFIDF_BASELINE:.4f})")
    else:
        print(f"TF-IDF остаётся лучшим ({NDCG_TFIDF_BASELINE:.4f}).")
        print(f"Лучший BERT: {best['Model']}  NDCG={best['NDCG']:.4f}")


Лучший пайплайн сохранён: pipeline_catboost_finetune_sim_labse_en_ru.pkl
NDCG: 0.7821  (TF-IDF: 0.7817)


In [45]:
def show_vacancy_predictions(pipeline, X_test, y_test, df,
                             n_top=10, vacancy_id=None, random_state=None):
    """
    Показывает предсказания пайплайна для выбранной вакансии из тест-сета.

    Args:
        pipeline     : обученный sklearn Pipeline
        X_test       : тестовые признаки (те же колонки, на которых обучался pipeline)
        y_test       : истинные метки
        df           : исходный датафрейм со всеми колонками (для отображения)
        n_top        : сколько топ-резюме показать
        vacancy_id   : None = случайная вакансия; иначе конкретный ID
        random_state : seed для воспроизводимости случайного выбора

    Returns:
        pd.DataFrame, отсортированный по pred_proba убыванием
    """
    df_test = df.loc[X_test.index].copy()
    df_test['target'] = y_test.values

    if vacancy_id is None:
        rng = np.random.RandomState(random_state)
        vacancy_id = rng.choice(df_test['vacancy_id'].unique())

    mask = df_test['vacancy_id'] == vacancy_id
    if not mask.any():
        print(f"[!] vacancy_id={vacancy_id} не найден в тестовой выборке.")
        return None

    df_vac = df_test[mask].copy()
    X_vac  = X_test.loc[df_vac.index]

    df_vac['pred_proba'] = pipeline.predict_proba(X_vac)[:, 1]
    df_vac = df_vac.sort_values('pred_proba', ascending=False).reset_index(drop=True)

    r0       = df_vac.iloc[0]
    vac_name = str(r0.get('vacancy_name', '—'))
    vac_desc = str(r0.get('vacancy_description', '—'))
    SEP, SEP2 = "=" * 90, "─" * 90

    print(SEP)
    print(f"  ВАКАНСИЯ : {vac_name}   [id={vacancy_id}]")
    print(f"  Опыт     : {r0.get('vacancy_experience','—')}  |  "
          f"Занятость : {r0.get('vacancy_employment','—')}  |  "
          f"График : {r0.get('vacancy_schedule','—')}")
    print(SEP2)
    print("  ОПИСАНИЕ ВАКАНСИИ:")
    for line in vac_desc[:1200].split('\n')[:25]:
        if line.strip():
            print(f"    {line.strip()}")
    if len(vac_desc) > 1200:
        print("    [... сокращено]")
    print(SEP)

    n_pos = int(df_vac['target'].sum())
    print(f"\n  ТОП-{n_top} из {len(df_vac)} кандидатов  "
          f"(всего реально подходящих в выборке: {n_pos})\n")

    for rank, (_, row) in enumerate(df_vac.head(n_top).iterrows(), 1):
        icon   = "✅" if row['target'] == 1 else "❌"
        exp    = str(row.get('resume_last_experience_description', '—'))
        skills = str(row.get('resume_skills', '—'))
        print(SEP2)
        print(f"  #{rank}  {icon}  score={row['pred_proba']:.4f}  "
              f"target={int(row['target'])}  [resume_id={row.get('resume_id','—')}]")
        print(f"  Должность : {row.get('resume_last_position','—')}")
        print(f"  Локация   : {row.get('resume_location','—')}  |  "
              f"Опыт: {row.get('resume_experience_months','—')} мес.")
        print(f"  Навыки    : {skills[:180]}{'...' if len(skills) > 180 else ''}")
        print(f"  Описание последнего места:")
        for line in exp[:700].split('\n')[:14]:
            if line.strip():
                print(f"    {line.strip()}")
        if len(exp) > 700:
            print("    [... сокращено]")
        print()

    print(SEP2)
    n_hit = int(df_vac.head(n_top)['target'].sum())
    print(f"\n  Попаданий в топ-{n_top}: {n_hit}/{n_top} ({100*n_hit//n_top}%)")
    print(f"  Всего релевантных в тест-сете для этой вакансии: {n_pos}\n")

    return df_vac

In [46]:
# ── Лучший пайплайн (включая fine-tuned) ─────────────────────────────────────
valid = [r for r in all_results if r.get('NDCG') is not None]
best  = max(valid, key=lambda r: r['NDCG'])
sim_col_best = best['sim_col']

X_test_best = X_test[BASE_FEATURES + ['als_score']].copy()
X_test_best[sim_col_best] = df.loc[X_test.index, sim_col_best].values

print(f"Используемый пайплайн: {best['Model']}  NDCG={best['NDCG']:.4f}")
result = show_vacancy_predictions(best['Pipeline'], X_test_best, y_test, df,
                                  n_top=10, random_state=42)

# ── Сравнить fine-tuned vs baseline на той же вакансии: ─────────────────────
# vacancy_id_fixed = result.iloc[0]['vacancy_id']
# r_base = next(r for r in all_results if 'LaBSE-en-ru' in r['Model'] and 'finetuned' not in r.get('sim_col',''))
# X_base = X_test[BASE_FEATURES + ['als_score']].copy()
# X_base[r_base['sim_col']] = df.loc[X_test.index, r_base['sim_col']].values
# print("\n--- BASELINE LaBSE ---")
# show_vacancy_predictions(r_base['Pipeline'], X_base, y_test, df,
#                          n_top=10, vacancy_id=vacancy_id_fixed)

Используемый пайплайн: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2  NDCG=0.7820
  ВАКАНСИЯ : Интернет-Маркетолог в Wildcrm   [id=126372900]
  Опыт     : От 3 до 6 лет  |  Занятость : Полная занятость  |  График : Удаленная работа
──────────────────────────────────────────────────────────────────────────────────────────
  ОПИСАНИЕ ВАКАНСИИ:
    О компании WildCRM — новый B2B-продукт для оцифровки финансовых данных маркетплейсов. Мы работаем с крупнейшими представителями рынка, такими как Wildberries и Ozon. Упрощаем финансовую отчетность и даем бизнесу возможность принимать data-driven решения. Мы хотим вырастить тебя с миддла до лида, который сможет возглавить направление интернет-маркетинга!   С нас:  Доход 130к- 150к Полная удаленка Рост от эксперта до лида команды Минимум бюрократии Поддержка коллег-маркетологов Команда исполнителей Атмосфера стартапа, стабильность зрелой компании    С тебя:  Запускать и масштабировать Telegram Ads Настраивать сквозную аналитику Быть